# နမူနာ ၀၂: OpenAI SDK ပေါင်းစည်းမှု

ဒီ notebook က OpenAI Python SDK နဲ့ အဆင့်မြင့်ပေါင်းစည်းမှုကို ပြသထားပြီး Microsoft Foundry Local နဲ့ Azure OpenAI နှစ်ခုလုံးကို streaming response နဲ့ မှန်ကန်တဲ့ error ကိုင်တွယ်မှုအတွက် ပံ့ပိုးပေးထားပါတယ်။

## အကျဉ်းချုပ်

ဒီနမူနာမှာ ပြသထားတာတွေက:
- Foundry Local နဲ့ Azure OpenAI အကြား အဆင်ပြေပြေ ပြောင်းလဲနိုင်မှု
- အသုံးပြုသူအတွေ့အကြုံ ပိုမိုကောင်းမွန်စေဖို့ streaming chat completions
- FoundryLocalManager SDK ကို မှန်ကန်စွာ အသုံးပြုမှု
- ခိုင်မာတဲ့ error ကိုင်တွယ်မှုနဲ့ fallback mechanism
- ထုတ်လုပ်မှုအဆင့် code ပုံစံများ


## အခြေခံလိုအပ်ချက်များ

- **Foundry Local**: တင်သွင်းပြီး အလုပ်လုပ်နေသည် (ဒေသတွင်းအတုယူမှုအတွက်)
- **Python**: 3.8 သို့မဟုတ် အထက်ရှိ OpenAI SDK ဖြင့်
- **Azure OpenAI**: သက်တမ်းရှိသော endpoint နှင့် API key (မိုဃ်းတိမ်အတုယူမှုအတွက်)

### အခြေခံလိုအပ်ချက်များကို တင်သွင်းပါ


In [ ]:
# Install required packages
!pip install openai foundry-local-sdk

## စာကြောင်းများကို တင်သွင်းပြီး Setup ပြုလုပ်ပါ


In [ ]:
import os
import sys
from openai import OpenAI
import time
from typing import Tuple

try:
    from foundry_local import FoundryLocalManager
    FOUNDRY_SDK_AVAILABLE = True
    print("✅ Foundry Local SDK is available")
except ImportError:
    FOUNDRY_SDK_AVAILABLE = False
    print("⚠️ Foundry Local SDK not available, manual configuration will be used")

## ဖွဲ့စည်းမှုရွေးချယ်မှုများ

သင့်အနေဖြင့် Azure OpenAI (Cloud) သို့မဟုတ် Foundry Local (On-device) ကို ရွေးချယ်ရန် သင့်အနေဖြင့် သင့်လိုအပ်သော ပတ်ဝန်းကျင်အပြောင်းအလဲများကို သတ်မှတ်ပါ။


### အရွေးချယ်မှု ၁: Azure OpenAI ဖွဲ့စည်းမှု

သင့် Azure OpenAI အထောက်အထားများကို Uncomment လုပ်ပြီး သတ်မှတ်ပါ:


In [ ]:
# Azure OpenAI Configuration
# Uncomment and set your actual values

# os.environ["AZURE_OPENAI_ENDPOINT"] = "https://your-resource.openai.azure.com"
# os.environ["AZURE_OPENAI_API_KEY"] = "your-api-key-here"
# os.environ["AZURE_OPENAI_API_VERSION"] = "2024-08-01-preview"
# os.environ["MODEL"] = "your-deployment-name"  # e.g., "gpt-4"

print("Azure OpenAI configuration ready (if credentials are set)")

### ရွေးချယ်မှု ၂: Foundry ဒေသခံကွန်ဖစ်ဂျူရေးရှင်း

ဒေသခံအချက်အလက်ခွဲခြမ်းစိတ်ဖြာမှုအတွက် ပုံမှန်ဆက်တင်များ:


In [ ]:
# Foundry Local Configuration (default)
FOUNDRY_MODEL = "phi-4-mini"  # Change to your preferred model
FOUNDRY_BASE_URL = "http://localhost:51211"
FOUNDRY_API_KEY = ""  # Usually empty for local

print(f"Foundry Local configuration ready with model: {FOUNDRY_MODEL}")

## Client Factory Functions

ဒီ function တွေက သင့် configuration အပေါ်မူတည်ပြီး သင့်အတွက် သင့်တော်တဲ့ OpenAI client ကို ဖန်တီးပေးပါသည်။


In [ ]:
def create_azure_client() -> Tuple[OpenAI, str]:
    """Create Azure OpenAI client."""
    azure_endpoint = os.environ.get("AZURE_OPENAI_ENDPOINT")
    azure_api_key = os.environ.get("AZURE_OPENAI_API_KEY")
    azure_api_version = os.environ.get("AZURE_OPENAI_API_VERSION", "2024-08-01-preview")
    
    if not azure_endpoint or not azure_api_key:
        raise ValueError("Azure OpenAI endpoint and API key are required")
    
    model = os.environ.get("MODEL", "your-deployment-name")
    client = OpenAI(
        base_url=f"{azure_endpoint}/openai",
        api_key=azure_api_key,
        default_query={"api-version": azure_api_version},
    )
    
    print(f"🌐 Azure OpenAI client created with model: {model}")
    return client, model


def create_foundry_client() -> Tuple[OpenAI, str]:
    """Create Foundry Local client with SDK management."""
    alias = FOUNDRY_MODEL
    
    if FOUNDRY_SDK_AVAILABLE:
        try:
            # Use FoundryLocalManager for proper service management
            print(f"🔄 Initializing Foundry Local with model: {alias}...")
            manager = FoundryLocalManager(alias)
            model_info = manager.get_model_info(alias)
            
            # Configure OpenAI client to use local Foundry service
            client = OpenAI(
                base_url=manager.endpoint,
                api_key=manager.api_key  # API key is not required for local usage
            )
            
            print(f"✅ Foundry Local SDK initialized")
            print(f"   Endpoint: {manager.endpoint}")
            print(f"   Model: {model_info.id}")
            return client, model_info.id
        except Exception as e:
            print(f"⚠️ Could not use Foundry SDK ({e}), falling back to manual configuration")
    
    # Fallback to manual configuration
    client = OpenAI(
        base_url=f"{FOUNDRY_BASE_URL}/v1",
        api_key=FOUNDRY_API_KEY
    )
    
    print(f"🔧 Manual Foundry Local configuration")
    print(f"   Endpoint: {FOUNDRY_BASE_URL}/v1")
    print(f"   Model: {alias}")
    return client, alias

## Client ကို Initialize လုပ်ပါ

ဒီဟာက Azure OpenAI သို့မဟုတ် Foundry Local ကို အသုံးပြုရမယ်ဆိုတာကို အလိုအလျောက် ရှာဖွေသိရှိပေးပါသည်။


In [ ]:
def initialize_client() -> Tuple[OpenAI, str, str]:
    """Initialize the appropriate OpenAI client."""
    
    # Check for Azure OpenAI configuration
    azure_endpoint = os.environ.get("AZURE_OPENAI_ENDPOINT")
    azure_api_key = os.environ.get("AZURE_OPENAI_API_KEY")
    
    if azure_endpoint and azure_api_key:
        print("🌐 Azure OpenAI configuration detected")
        try:
            client, model = create_azure_client()
            return client, model, "azure"
        except Exception as e:
            print(f"❌ Azure OpenAI initialization failed: {e}")
            print("🔄 Falling back to Foundry Local...")
    
    # Use Foundry Local
    print("🏠 Using Foundry Local configuration")
    try:
        client, model = create_foundry_client()
        return client, model, "foundry"
    except Exception as e:
        print(f"❌ Foundry Local initialization failed: {e}")
        raise

# Initialize the client
print("Initializing OpenAI client...")
print("=" * 50)
client, model, provider = initialize_client()
print("=" * 50)
print(f"✅ Initialization complete! Using {provider} with model: {model}")

## အခြေခံ Chat Completion

အခြေခံ chat completion ကို စမ်းသပ်ပါ:


In [ ]:
def simple_chat(prompt: str, max_tokens: int = 150) -> str:
    """Send a simple chat message and get response."""
    try:
        response = client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": prompt}],
            max_tokens=max_tokens
        )
        return response.choices[0].message.content
    except Exception as e:
        return f"Error: {e}"

# Test basic chat
test_prompt = "Say hello from the SDK quickstart and explain what you are in one sentence."

print(f"👤 User: {test_prompt}")
print("\n🤖 Assistant:")
response = simple_chat(test_prompt)
print(response)

## စီးဆင်းမှု Chat Completion

အသုံးပြုသူအတွေ့အကြုံပိုမိုကောင်းမွန်စေရန် စီးဆင်းမှုအဖြေများကို ပြသပါ:


In [ ]:
def streaming_chat(prompt: str, max_tokens: int = 300) -> str:
    """Send a chat message with streaming response."""
    try:
        print("🤖 Assistant (streaming):")
        
        stream = client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": prompt}],
            max_tokens=max_tokens,
            stream=True
        )
        
        full_response = ""
        for chunk in stream:
            if chunk.choices[0].delta.content is not None:
                content = chunk.choices[0].delta.content
                print(content, end="", flush=True)
                full_response += content
        
        print("\n")  # New line after streaming
        return full_response
    except Exception as e:
        error_msg = f"Error: {e}"
        print(error_msg)
        return error_msg

# Test streaming chat
streaming_prompt = "Explain the key benefits of using Microsoft Foundry Local for AI development. Include aspects like privacy, performance, and cost."

print(f"👤 User: {streaming_prompt}\n")
streaming_response = streaming_chat(streaming_prompt)

## မျိုးစုံအကြိမ်ဆွေးနွေးမှု

ဆွေးနွေးမှုအကြောင်းအရာကို ဆက်လက်ထိန်းသိမ်းထားနိုင်မှုကို ပြသပါ:


In [ ]:
class ConversationManager:
    """Manages multi-turn conversations with context."""
    
    def __init__(self, system_prompt: str = None):
        self.messages = []
        if system_prompt:
            self.messages.append({"role": "system", "content": system_prompt})
    
    def send_message(self, user_message: str, max_tokens: int = 200) -> str:
        """Send a message and get response while maintaining context."""
        # Add user message to conversation
        self.messages.append({"role": "user", "content": user_message})
        
        try:
            response = client.chat.completions.create(
                model=model,
                messages=self.messages,
                max_tokens=max_tokens
            )
            
            assistant_message = response.choices[0].message.content
            
            # Add assistant response to conversation
            self.messages.append({"role": "assistant", "content": assistant_message})
            
            return assistant_message
        except Exception as e:
            return f"Error: {e}"
    
    def get_conversation_length(self) -> int:
        """Get the number of messages in the conversation."""
        return len(self.messages)

# Create conversation manager with system prompt
system_prompt = "You are a helpful AI assistant specialized in explaining AI and machine learning concepts. Be concise but informative."
conversation = ConversationManager(system_prompt)

# Multi-turn conversation example
conversation_turns = [
    "What is the difference between AI inference on-device vs in the cloud?",
    "Which approach is better for privacy?",
    "What about performance and latency considerations?"
]

for i, turn in enumerate(conversation_turns, 1):
    print(f"\n{'='*60}")
    print(f"Turn {i}")
    print(f"{'='*60}")
    print(f"👤 User: {turn}")
    
    response = conversation.send_message(turn)
    print(f"\n🤖 Assistant: {response}")

print(f"\n📊 Conversation summary: {conversation.get_conversation_length()} messages total")

## စွမ်းဆောင်ရည်နှိုင်းယှဉ်ခြင်း

အခြေအနေများအမျိုးမျိုးအတွက် တုံ့ပြန်ချိန်များကို နှိုင်းယှဉ်ပါ:


In [ ]:
def benchmark_response_time(prompt: str, iterations: int = 3) -> dict:
    """Benchmark response time for a given prompt."""
    times = []
    responses = []
    
    for i in range(iterations):
        start_time = time.time()
        
        try:
            response = client.chat.completions.create(
                model=model,
                messages=[{"role": "user", "content": prompt}],
                max_tokens=50  # Keep responses short for timing
            )
            
            end_time = time.time()
            response_time = end_time - start_time
            
            times.append(response_time)
            responses.append(response.choices[0].message.content)
            
        except Exception as e:
            print(f"Error in iteration {i+1}: {e}")
    
    if times:
        avg_time = sum(times) / len(times)
        min_time = min(times)
        max_time = max(times)
        
        return {
            "average_time": avg_time,
            "min_time": min_time,
            "max_time": max_time,
            "all_times": times,
            "sample_response": responses[0] if responses else None
        }
    
    return {"error": "No successful responses"}

# Benchmark different types of prompts
benchmark_prompts = [
    "What is AI?",
    "Explain machine learning in simple terms.",
    "List 3 benefits of edge computing."
]

print(f"⏱️  Performance Benchmark ({provider} - {model})")
print("=" * 60)

for prompt in benchmark_prompts:
    print(f"\n📝 Prompt: '{prompt}'")
    results = benchmark_response_time(prompt)
    
    if "error" not in results:
        print(f"   ⏰ Average time: {results['average_time']:.2f}s")
        print(f"   ⚡ Fastest: {results['min_time']:.2f}s")
        print(f"   🐌 Slowest: {results['max_time']:.2f}s")
        print(f"   📄 Sample response: {results['sample_response'][:100]}...")
    else:
        print(f"   ❌ {results['error']}")

## အဆင့်မြင့် Configuration နှင့် Error ကိုင်တွယ်မှု

Parameter အမျိုးမျိုးနှင့် Error ဖြစ်နိုင်ခြေများကို စမ်းသပ်ပါ:


In [ ]:
def test_different_parameters():
    """Test chat completions with different parameters."""
    prompt = "Write a creative short story about AI."
    
    # Test different temperature values
    temperatures = [0.1, 0.5, 0.9]
    
    for temp in temperatures:
        print(f"\n🌡️ Temperature: {temp}")
        print("-" * 30)
        
        try:
            response = client.chat.completions.create(
                model=model,
                messages=[{"role": "user", "content": prompt}],
                max_tokens=100,
                temperature=temp
            )
            
            print(f"Response: {response.choices[0].message.content[:150]}...")
            
        except Exception as e:
            print(f"Error with temperature {temp}: {e}")

test_different_parameters()

## ဝန်ဆောင်မှု ကျန်းမာရေး စစ်ဆေးမှု

ဝန်ဆောင်မှု ကျန်းမာရေးနှင့် စွမ်းဆောင်ရည် စစ်ဆေးမှု အပြည့်အစုံ:


In [ ]:
def comprehensive_health_check():
    """Perform comprehensive health check of the service."""
    print("🏥 Comprehensive Health Check")
    print("=" * 50)
    
    # 1. Check model listing
    try:
        models_response = client.models.list()
        available_models = [m.id for m in models_response.data]
        print(f"✅ Model listing: SUCCESS")
        print(f"   📋 Available models: {available_models}")
        
        if model in available_models:
            print(f"   ✅ Current model '{model}' is available")
        else:
            print(f"   ⚠️ Current model '{model}' not found in available models")
    except Exception as e:
        print(f"❌ Model listing: FAILED - {e}")
    
    # 2. Test basic completion
    try:
        test_response = client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": "Say 'Health check successful'"}],
            max_tokens=10
        )
        print(f"✅ Basic completion: SUCCESS")
        print(f"   💬 Response: {test_response.choices[0].message.content}")
    except Exception as e:
        print(f"❌ Basic completion: FAILED - {e}")
    
    # 3. Test streaming
    try:
        stream = client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": "Count to 3"}],
            max_tokens=20,
            stream=True
        )
        
        stream_content = ""
        chunk_count = 0
        for chunk in stream:
            if chunk.choices[0].delta.content:
                stream_content += chunk.choices[0].delta.content
                chunk_count += 1
        
        print(f"✅ Streaming: SUCCESS")
        print(f"   📦 Chunks received: {chunk_count}")
        print(f"   💬 Streamed content: {stream_content.strip()}")
    except Exception as e:
        print(f"❌ Streaming: FAILED - {e}")
    
    # 4. Provider-specific information
    print(f"\n📊 Configuration Summary:")
    print(f"   🏢 Provider: {provider}")
    print(f"   🤖 Model: {model}")
    if provider == "foundry":
        print(f"   🏠 Foundry SDK Available: {FOUNDRY_SDK_AVAILABLE}")
        print(f"   🔗 Base URL: {FOUNDRY_BASE_URL}")
    elif provider == "azure":
        print(f"   🌐 Azure Endpoint: {os.environ.get('AZURE_OPENAI_ENDPOINT', 'Not set')}")
        print(f"   🔑 API Version: {os.environ.get('AZURE_OPENAI_API_VERSION', 'Not set')}")

comprehensive_health_check()

## အပြန်အလှန် စမ်းသပ်ခြင်း

ဤဆဲလ်ကို သင့်ကိုယ်ပိုင် prompt များကို အပြန်အလှန်စမ်းသပ်ရန် အသုံးပြုပါ:


In [ ]:
# Interactive testing - modify the prompt below
custom_prompt = "Explain the concept of 'edge AI' and why it's becoming important."
use_streaming = True  # Set to False for regular completion

print(f"👤 Custom Prompt: {custom_prompt}\n")

if use_streaming:
    custom_response = streaming_chat(custom_prompt, max_tokens=250)
else:
    custom_response = simple_chat(custom_prompt, max_tokens=250)
    print(f"🤖 Assistant: {custom_response}")

## အကျဉ်းချုပ်နှင့် နောက်ထပ်အဆင့်များ

ဒီ notebook မှာ OpenAI SDK ကို အဆင့်မြင့်ပေါင်းစည်းမှုအကြောင်းကို ပြသခဲ့ပါတယ်။

### ✅ အဓိကအချက်များ

1. **Multi-Provider Support**: Azure OpenAI နဲ့ Foundry Local အကြား အလွယ်တကူ ပြောင်းလဲနိုင်မှု
2. **Streaming Responses**: UX ကိုပိုမိုကောင်းမွန်စေဖို့ အချိန်နှင့်တပြေးညီ token ထုတ်ပေးမှု
3. **Conversation Management**: အကြောင်းအရာကို ထိန်းသိမ်းထားပြီး Multi-turn စကားဝိုင်းများ
4. **Performance Benchmarking**: တုံ့ပြန်မှုအချိန်ကို တိုင်းတာခြင်းနှင့် ခွဲခြမ်းစိတ်ဖြာခြင်း
5. **Comprehensive Health Checks**: ဝန်ဆောင်မှုအတည်ပြုခြင်းနှင့် အကဲဖြတ်မှု
6. **Error Handling**: အမှားများကို ချုပ်ကိုင်နိုင်သော စနစ်နှင့် fallback များ

### 🏆 Foundry Local နှင့် Azure OpenAI

| Aspect | Foundry Local | Azure OpenAI |
|--------|---------------|---------------|
| **Privacy** | ✅ ဒေတာကို ဒေသတွင်းမှာသာထားရှိ | ⚠️ Cloud ကို ဒေတာပို့ပေးရ |
| **Latency** | ✅ အနည်းဆုံး (local inference) | ⚠️ ပိုများ (network အားပေါ်မူတည်) |
| **Cost** | ✅ အခမဲ့ (hardware အပြီး) | 💰 Token အလိုက် ပေးချေမှု |
| **Offline** | ✅ အင်တာနက်မလိုအပ် | ❌ အင်တာနက်လိုအပ် |
| **Model Variety** | ⚠️ ရွေးချယ်မှု အကန့်အသတ်ရှိ | ✅ Model အားလုံးရရှိနိုင် |
| **Scaling** | ⚠️ Hardware အားပေါ်မူတည် | ✅ အကန့်အသတ်မရှိသော scalability |

### 🚀 နောက်ထပ်အဆင့်များ

- **Sample 04**: Chainlit chat application တည်ဆောက်ခြင်း
- **Sample 05**: Multi-agent orchestration systems
- **Sample 06**: Intelligent model routing
- **Production Deployment**: Scaling နှင့် monitoring အရေးယူမှုများ

### 💡 အကောင်းဆုံးအလေ့အကျင့်များ

1. **Provider များအကြား fallback mechanisms** ကို အမြဲတမ်းထည့်သွင်းပါ
2. **Streaming ကို အသုံးပြုပါ** အကြမ်းမဲ့သော တုံ့ပြန်မှုများအတွက် UX ကိုတိုးတက်စေ
3. **Error handling ကို သေချာစွာ အကောင်အထည်ဖော်ပါ** production applications အတွက်
4. **Response time နှင့် cost များကို စောင့်ကြည့်ပါ** provider အလိုက်
5. **သင့်လိုအပ်ချက်အပေါ်မူတည်ပြီး provider ကို ရွေးချယ်ပါ**
